# 01 — Phase 1: Baseline Circuit Mapping

For every **unambiguous** prompt, we:
1. Run the model with `run_with_cache` to capture all activations
2. Compute the logit difference (answer_A − answer_B) — our primary scalar metric
3. Run **activation patching** for every head (12 layers × 12 heads = 144 heads):
   patch each head's `z` activation from the clean (unambiguous) run into a
   corrupted (opposite-answer unambiguous) run, and record the logit-diff change
4. Save per-prompt heatmaps (layer × head) to `data/results/phase1/`

**Outputs**
- `data/results/phase1/contributions_{prompt_id}.npy` — (n_layers, n_heads) array
- `data/results/phase1/phase1_summary.csv` — one row per unambiguous prompt

**Expected runtime:** ~2–4 hours on M5 Pro (144 hook-runs per prompt)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

from circuit_conflict.utils import load_model, get_device, get_answer_token_id, logit_diff
from circuit_conflict.dataset import load_prompts
from circuit_conflict.patching import head_contribution_map

DEVICE = get_device()
RESULTS_DIR = Path('../data/results/phase1')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
model = load_model(DEVICE)
df_all = load_prompts()

## 1. Split unambiguous prompts into paired groups

Each unambiguous prompt is the "clean" run. Its partner (with the *other* answer
favoured) is the "corrupted" run used for activation patching.

Pairing convention: for a pair `X_NNN_conflict` / `X_NNN_unamb`,  
the clean run = unambiguous version that matches ground_truth,  
the corrupted run = the unambiguous version flipped (we use the conflict prompt as corrupted,  
which is the most natural 'corrupted' baseline).

In [ ]:
# Unambiguous rows only (our "clean" baselines)
df_unamb = df_all[~df_all['is_conflict']].reset_index(drop=True)
# Conflict rows (used as the "corrupted" baseline in activation patching)
df_conflict = df_all[df_all['is_conflict']].reset_index(drop=True)

print(f'Unambiguous prompts: {len(df_unamb)}')
print(f'Conflict prompts   : {len(df_conflict)}')

## 2. Activation patching loop

For each unambiguous prompt:
- **clean** = unambiguous tokens
- **corrupted** = corresponding conflict-version tokens (same answer tokens A/B)

Patch every head from clean → corrupted and record Δ logit_diff.

In [ ]:
summary_rows = []

for _, urow in tqdm(df_unamb.iterrows(), total=len(df_unamb), desc='Phase 1 patching'):
    pid = urow['prompt_id']  # e.g. "A_001_unamb"
    pair_id = pid.replace('_unamb', '_conflict')
    
    # Find paired conflict row
    conflict_matches = df_conflict[df_conflict['prompt_id'] == pair_id]
    if len(conflict_matches) == 0:
        print(f'  Warning: no conflict partner for {pid}, skipping')
        continue
    crow = conflict_matches.iloc[0]

    # Get token IDs for the two answers
    try:
        tok_A = get_answer_token_id(model, urow['answer_A'])
        tok_B = get_answer_token_id(model, urow['answer_B'])
    except ValueError as e:
        print(f'  Skipping {pid}: {e}')
        continue

    tokens_clean = model.to_tokens(urow['prompt_text'])
    tokens_corrupted = model.to_tokens(crow['prompt_text'])

    # Baseline logit diff on the clean run
    with torch.no_grad():
        logits_clean = model(tokens_clean)
    diff_clean = logit_diff(logits_clean, tok_A, tok_B)

    # Head contribution map (144 hook-patching runs)
    contrib = head_contribution_map(
        model, tokens_clean, tokens_corrupted, tok_A, tok_B
    )
    
    # Save contribution array
    out_path = RESULTS_DIR / f'contributions_{pid}.npy'
    np.save(out_path, contrib)

    summary_rows.append({
        'prompt_id': pid,
        'category': urow['category'],
        'ground_truth': urow['ground_truth'],
        'logit_diff_clean': diff_clean,
        'top_layer': int(np.unravel_index(np.argmax(contrib), contrib.shape)[0]),
        'top_head': int(np.unravel_index(np.argmax(contrib), contrib.shape)[1]),
        'top_contribution': float(contrib.max()),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(RESULTS_DIR / 'phase1_summary.csv', index=False)
print(f'\nSaved summary for {len(df_summary)} prompts')
df_summary.head()

## 3. Build average Circuit Maps per category

Average the (n_layers, n_heads) contribution arrays across all prompts in each
category to produce stable circuit maps.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

N_LAYERS = model.cfg.n_layers  # 12
N_HEADS  = model.cfg.n_heads   # 12

for cat in ['A', 'B', 'C']:
    cat_rows = df_summary[df_summary['category'] == cat]
    if len(cat_rows) == 0:
        print(f'No data for category {cat}')
        continue

    # Load and average contribution arrays
    arrays = []
    for _, row in cat_rows.iterrows():
        p = RESULTS_DIR / f"contributions_{row['prompt_id']}.npy"
        if p.exists():
            arrays.append(np.load(p))
    if not arrays:
        continue
    mean_contrib = np.stack(arrays).mean(axis=0)  # (n_layers, n_heads)

    # Save the averaged map
    np.save(RESULTS_DIR / f'circuit_map_cat{cat}.npy', mean_contrib)

    # Plot
    fig, ax = plt.subplots(figsize=(9, 6))
    vmax = max(abs(mean_contrib.max()), abs(mean_contrib.min()))
    im = ax.imshow(mean_contrib, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto', interpolation='nearest')
    ax.set_xlabel('Head')
    ax.set_ylabel('Layer')
    ax.set_xticks(range(N_HEADS))
    ax.set_yticks(range(N_LAYERS))
    ax.set_title(f'Category {cat}: Mean Head Causal Contribution\n'
                 f'(n={len(arrays)} unambiguous prompts, '
                 f'Δ logit_diff from activation patch)')
    plt.colorbar(im, ax=ax, label='Δ logit_diff (A−B)')
    plt.tight_layout()
    plt.savefig(f'../figures/01_circuit_map_cat{cat}.png', dpi=150)
    plt.show()
    print(f'Category {cat}: top head = layer {mean_contrib.argmax() // N_HEADS}, '
          f'head {mean_contrib.argmax() % N_HEADS}, '
          f'score={mean_contrib.max():.4f}')

## 4. Identify the top Circuit A and Circuit B heads

We call a head part of **Circuit A** if its mean contribution is strongly positive
(favours answer_A), and part of **Circuit B** if it's strongly negative.

In [ ]:
# Overall average across all categories
all_contrib_files = sorted(RESULTS_DIR.glob('contributions_*.npy'))
all_arrays = np.stack([np.load(f) for f in all_contrib_files])  # (n_prompts, n_layers, n_heads)
overall_mean = all_arrays.mean(axis=0)  # (n_layers, n_heads)
np.save(RESULTS_DIR / 'circuit_map_overall.npy', overall_mean)

flat_sorted = sorted(
    [(l, h, overall_mean[l, h]) for l in range(N_LAYERS) for h in range(N_HEADS)],
    key=lambda x: abs(x[2]), reverse=True
)

print('Top 10 heads by absolute causal contribution (both circuits):')
print(f'{"Rank":>4}  {"Layer":>5}  {"Head":>4}  {"Score":>8}')
for rank, (l, h, s) in enumerate(flat_sorted[:10], 1):
    circuit = 'Circuit A' if s > 0 else 'Circuit B'
    print(f'{rank:>4}  {l:>5}  {h:>4}  {s:>8.4f}  {circuit}')

In [ ]:
# Save the top-head lists for reference in Phase 3
circuit_A_heads = [(l, h) for l, h, s in flat_sorted if s > 0][:20]
circuit_B_heads = [(l, h) for l, h, s in flat_sorted if s < 0][:20]

import json
with open(RESULTS_DIR / 'circuit_A_heads.json', 'w') as f:
    json.dump(circuit_A_heads, f)
with open(RESULTS_DIR / 'circuit_B_heads.json', 'w') as f:
    json.dump(circuit_B_heads, f)

print(f'Circuit A top-20 heads saved.')
print(f'Circuit B top-20 heads saved.')
print('\nPhase 1 complete. ✓')